# JOUR 701 — Week 11 Day 1 Follow-Along
## Correlation, t-test, and simple linear regression

**Same dataset, next step.**  
Last week: prepare and clean the data.  
This week: use the data to answer research questions.

You do **not** need to memorize the code. Run each cell and focus on what the result means.


## 1. Load the course dataset

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

url = "https://raw.githubusercontent.com/KayChansiri/Course_research-methods-in-the-age-of-ai/main/data/JOUR701_Block3_Course_Dataset.csv"
df = pd.read_csv(url)

df.head()


## 2. Quick check

We will use:
- `baseline_trust` — numeric
- `message_trust` — numeric
- `condition` — categorical


In [ ]:
df[["baseline_trust", "message_trust", "condition"]].head(10)


In [ ]:
df[["baseline_trust", "message_trust"]].describe()


In [ ]:
df[["baseline_trust", "message_trust", "condition"]].isna().sum()


# Part A — Correlation
### Question: Are baseline trust and message trust related?

Read:
- **sign of r** → direction
- **size of |r|** → strength
- **p-value** → evidence against a zero correlation


In [ ]:
from scipy.stats import pearsonr

corr_data = df[["baseline_trust", "message_trust"]].dropna()

r, p = pearsonr(
    corr_data["baseline_trust"],
    corr_data["message_trust"]
)

print("r =", round(r, 3))
print("p =", round(p, 4))


### Interpret it
1. The correlation is **positive / negative**.
2. **r = ____**
3. **p = ____**
4. In plain language: __________________________________________.

Do **not** say correlation proves causation.


In [ ]:
plt.scatter(corr_data["baseline_trust"], corr_data["message_trust"])
plt.xlabel("Baseline trust")
plt.ylabel("Message trust")
plt.title("Baseline Trust and Message Trust")
plt.show()


# Part B — t-test
### Question: Does average message trust differ between Control and Personalized?

- `condition` = two groups
- `message_trust` = numeric outcome


In [ ]:
from scipy.stats import ttest_ind

# Make the condition labels lowercase so the code works even if capitalization differs.
condition_lower = df["condition"].astype(str).str.strip().str.lower()

control = df.loc[condition_lower == "control", "message_trust"].dropna()
personalized = df.loc[condition_lower == "personalized", "message_trust"].dropna()

print("Control n:", len(control))
print("Personalized n:", len(personalized))
print("Control mean:", round(control.mean(), 2))
print("Personalized mean:", round(personalized.mean(), 2))

t, p = ttest_ind(control, personalized, equal_var=False)

print("t =", round(t, 3))
print("p =", round(p, 4))


### Interpret it
Ask:
1. Which group has the higher mean?
2. How large is the difference?
3. Is p < .05?

Complete:

> Participants in the __________ condition had an average message trust score of ____, compared with ____ in the __________ condition. The t-test p-value was ____.


In [ ]:
df.boxplot(column="message_trust", by="condition")
plt.xlabel("Condition")
plt.ylabel("Message trust")
plt.title("Message Trust by Condition")
plt.suptitle("")
plt.show()


# Part C — Simple linear regression
### Question: Does baseline trust predict message trust?

- **X / predictor** = `baseline_trust`
- **Y / outcome** = `message_trust`


In [ ]:
import statsmodels.formula.api as smf

model = smf.ols(
    "message_trust ~ baseline_trust",
    data=df
).fit()

model.summary()


## Do NOT read every number in the regression table

Find:
1. **coef** for `baseline_trust`
2. **P>|t|**
3. **[0.025, 0.975]**
4. **R-squared**


In [ ]:
coef = model.params["baseline_trust"]
p_value = model.pvalues["baseline_trust"]
ci_low, ci_high = model.conf_int().loc["baseline_trust"]
r_squared = model.rsquared

print("Coefficient:", round(coef, 3))
print("p-value:", round(p_value, 4))
print("95% CI:", [round(ci_low, 3), round(ci_high, 3)])
print("R-squared:", round(r_squared, 3))


### Interpret the coefficient

> A 1-point increase in baseline trust is associated with an estimated **____-point increase/decrease** in message trust.

Then check:
- positive or negative?
- does the 95% CI include 0?
- is p < .05?
- what is R²?

Do **not** automatically write “baseline trust causes message trust.”


In [ ]:
import numpy as np

plot_df = df[["baseline_trust", "message_trust"]].dropna()

plt.scatter(plot_df["baseline_trust"], plot_df["message_trust"])

m, b = np.polyfit(
    plot_df["baseline_trust"],
    plot_df["message_trust"],
    1
)

x = np.linspace(
    plot_df["baseline_trust"].min(),
    plot_df["baseline_trust"].max(),
    100
)

plt.plot(x, m*x + b)
plt.xlabel("Baseline trust")
plt.ylabel("Message trust")
plt.title("Simple Linear Regression")
plt.show()


# Final comparison

| Question | Variable setup | Tool |
|---|---|---|
| Are `baseline_trust` and `message_trust` related? | numeric + numeric | **Correlation** |
| Does `message_trust` differ by `condition`? | 2 groups + numeric outcome | **t-test** |
| Does `baseline_trust` predict `message_trust`? | predictor → numeric outcome | **Linear regression** |

## Rule to remember
**Question → variable types → analysis**


## Quick checkpoint
1. What does **r** tell you?
2. What does a **t-test** compare?
3. In regression, which variable is **Y**?
4. What does the regression **coefficient** tell you?
5. What does **R²** mean?
